# Lab 3 (by 2nd part of the course): "Bridging the gap between training a ML model and deploying it"

In this assignment, you will bridge the gap between Data Science and MLOps. You will train a simple Machine Learning model, wrap it in a REST API that processes requests via a task queue, and package the entire system into a Docker container.

By the end of this task, you will have a standalone microservice where users can submit data for prediction and asynchronously retrieve their results.

## System Architecture

Your service will implement an asynchronous POST and Receive (Polling) pattern:

1. Client POSTs data to a `/submit` endpoint.
2. Server places the task in a queue, assigns it a unique Task ID, and immediately returns the Task ID to the client.
3. ML Worker processes the queue in the background, running the data through the model.
4. Client GETs (requests) the `/result/{task_id}` endpoint to check the status. If the model has finished processing, the server returns the prediction.

## Step-by-Step Instructions
### Step 1: Train and Serialize a Simple Model

Write a Python script (`train.py`) to train a simple model.

1. Use scikit-learn to load a toy dataset or use any model from your previous lab works (I recommend table data).
2. Train a simple model (e.g., LogisticRegression or LinearRegression would be sufficient).
3. Save the trained model to a file using joblib or pickle (e.g., `model.pkl`).
4. Don't forget to fixate used libraries versions and data processing pipeline (if such required)

### Step 2: Create the Web API and Task Queue

Write your application script (e.g., `app.py`). You are encouraged to use FastAPI, as it makes building asynchronous web APIs straightforward.

1. Load the Model: Ensure your app loads model.pkl into memory when it starts up.
2. Setup the Queue: For this assignment, you can use Python's built-in asyncio background tasks and a global dictionary to store task statuses and results.
3. Create Endpoints:
    * POST `/predict`: Accepts input features, generates a unique UUID for the task, starts the prediction in the background, and returns {"task_id": "your-uuid", "status": "processing"}.
    * GET `/result/{task_id}`: Checks the dictionary for the given ID. Returns `{"status": "processing"}` if still running, or `{"status": "completed", "prediction": [...]}` if done.

### Step 3: Write the Dockerfile

Create a Dockerfile in the root of your project directory to containerize your application.

1. Start with a lightweight Python base image or with Ubuntu image (and install required Python and libraries).
2. Set your working directory (WORKDIR `/app`).
3. Copy your requirements.txt and install dependencies (`RUN pip install -r requirements.txt`).
4. Copy your serialized model (`model.pkl`) and application code (`app.py`) into the container.
5. Expose the port your API runs on (e.g., `EXPOSE 8000`).
6. Define the command to run your server (e.g., `CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]`).

### Step 4: Build and Run your Container

Open your terminal and use Docker CLI commands to bring your service to life.

1. Build the image: `docker build -t ml-async-service .`
2. Run the container: `docker run -d -p 8888:8888 -p 4444:22 ml-async-service`

### Step 5: Test the Service

Write a small test script (`test.py`) using the requests library, or use curl commands to prove your service works:

1. Send a POST request with sample data.
2. Extract the `task_id` from the response.
3. Write a loop that sends a GET request every 2 seconds to the result endpoint until the prediction is returned.